In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE DATABASE default")

In [0]:
pip install aiohttp


In [0]:
from repository.collection_repository import CollectionRepository

repo = CollectionRepository(spark)

df = spark.table(repo.table_name)

display(df.select("collection_name", "contract_address", "status"))

In [0]:
display(spark.table("nft_base_table"))

In [0]:
display(spark.table("nft_metadata_table"))

In [0]:
display(spark.table("nft_rarity_table"))

In [0]:
display(spark.table("nft_rarity_table").filter(col("collection_id") == 1))

In [0]:
%sql
SELECT *
FROM nft_rarity_table
ORDER BY rarity_score DESC
LIMIT 10

In [0]:
spark.table("collection_reference").show()

In [0]:
from pyspark.sql.functions import col

# ── Get token_ids from both tables ─────────────
base_tokens = spark.table("nft_base_table") \
    .filter(col("collection_id") == "2") \
    .select("token_id") \
    .distinct()

rarity_tokens = spark.table("nft_rarity_table") \
    .filter(col("collection_id") == "2") \
    .select("token_id") \
    .distinct()

# ── Check counts ───────────────────────────────
print(f"Base tokens  : {base_tokens.count()}")
print(f"Rarity tokens: {rarity_tokens.count()}")

# ── Find tokens in base but NOT in rarity ──────
in_base_not_rarity = base_tokens.subtract(rarity_tokens)
print(f"\nIn base but NOT in rarity: {in_base_not_rarity.count()}")
display(in_base_not_rarity)

# ── Find tokens in rarity but NOT in base ──────
in_rarity_not_base = rarity_tokens.subtract(base_tokens)
print(f"\nIn rarity but NOT in base: {in_rarity_not_base.count()}")
display(in_rarity_not_base)

# ── Show sample from both tables ───────────────
print("\n=== BASE TABLE sample ===")
display(
    spark.table("nft_base_table")
    .filter(col("collection_id") == "2")
    .orderBy("token_id")
    .limit(10)
)

print("\n=== RARITY TABLE sample ===")
display(
    spark.table("nft_rarity_table")
    .filter(col("collection_id") == "2")
    .orderBy("rank")
    .limit(10)
)

In [0]:
from pyspark.sql.functions import col

base_tokens   = spark.table("nft_base_table") \
    .filter(col("collection_id") == "2") \
    .select("token_id").distinct()

rarity_tokens = spark.table("nft_rarity_table") \
    .filter(col("collection_id") == "2") \
    .select("token_id").distinct()

print(f"Base tokens  : {base_tokens.count()}")
print(f"Rarity tokens: {rarity_tokens.count()}")

# Tokens in rarity but NOT in base ← these are the wrong ones
wrong_tokens = rarity_tokens.subtract(base_tokens)
print(f"Wrong tokens in rarity: {wrong_tokens.count()}")
display(wrong_tokens)

In [0]:
display(spark.table("nft_base_table"))

In [0]:
display(spark.table("nft_metadata_table"))

In [0]:
spark.table("nft_base_table").count()

In [0]:
spark.table("nft_rarity_table").display()

In [0]:
from pyspark.sql.functions import col
collection_ids = [1, 2]  # your collection IDs

for cid in collection_ids:
    print(f"\n===== BASE TABLE → Collection {cid} =====")
    display(
        spark.table("nft_base_table")
        .filter(col("collection_id") == str(cid))
    )

In [0]:
for cid in collection_ids:
    print(f"\n===== METADATA TABLE → Collection {cid} =====")
    display(
        spark.table("nft_metadata_table")
        .filter(col("collection_id") == str(cid))
    )

In [0]:
for cid in collection_ids:
    print(f"\n===== RARITY TABLE → Collection {cid} =====")
    display(
        spark.table("nft_rarity_table")
        .filter(col("collection_id") == str(cid))
    )

In [0]:
spark.table("nft_base_table") \
    .filter(col("collection_id") == "collection_1") \
    .select("token_id") \
    .distinct() \
    .count()

In [0]:
from pyspark.sql.functions import col

collection_id = "1"  # change to "2" for collection_2

# ── Base NFTs ──────────────────────────────────
display(spark.table("nft_base_table")
    .filter(col("collection_id") == collection_id)
    .orderBy(col("token_id").cast("int")))

# ── Metadata ───────────────────────────────────
display(spark.table("nft_metadata_table")
    .filter(col("collection_id") == collection_id)
    .orderBy(col("token_id").cast("int")))

# ── Rarity ─────────────────────────────────────
display(spark.table("nft_rarity_table")
    .filter(col("collection_id") == collection_id)
    .orderBy("rank"))

# ── Top 20 ─────────────────────────────────────
display(spark.table("nft_rarity_table")
    .filter(col("collection_id") == collection_id)
    .orderBy("rank")
    .limit(20))

In [0]:
import sys
sys.path.append("/Workspace/Repos/iamanmolsingh0019@gmail.com/ethereum-project/my_nft_project")

from manager.nft_collection_manager import NFTCollectionManager

ALCHEMY_KEY = dbutils.secrets.get(scope="nft-scope", key="ALCHEMY_KEY")
manager = NFTCollectionManager(spark, ALCHEMY_KEY)

# ── Then just call run() ───────────────────────
# await manager.run("0x23581767a106ae21c074b2276D25e5C3e136a68b")
await manager.run("0xBd3531dA5CF5857e7CfAA92426877b022e612cf8")

In [0]:
import os
print(os.getcwd())

In [0]:
from my_nft_project import get_manager

manager = get_manager(spark, dbutils) 

manager.run("0xC35e29179BA7BCE8b28ABBAb849F62C52bd2E88a")
manager.run("0xBd3531dA5CF5857e7CfAA92426877b022e612cf8")

In [0]:
from pyspark.sql.functions import col
contract = "0xC35e29179BA7BCE8b28ABBAb849F62C52bd2E88a"
display(spark.table("nft_base_table")
    .filter(col("contract_address") == contract)
    .orderBy(col("token_id").cast("int")))

# ── Metadata Table ─────────────────────────────
display(spark.table("nft_metadata_table")
    .filter(col("contract_address") == contract)
    .orderBy(col("token_id").cast("int")))

# ── Rarity Table ───────────────────────────────
collection_id = spark.table("nft_base_table") \
    .filter(col("contract_address") == contract) \
    .select("collection_id") \
    .first()["collection_id"]

display(spark.table("nft_rarity_table")
    .filter(col("collection_id") == collection_id)
    .orderBy("rank"))

In [0]:
from pyspark.sql.functions import col
contract = "0xBd3531dA5CF5857e7CfAA92426877b022e612cf8"
display(spark.table("nft_base_table")
    .filter(col("contract_address") == contract)
    .orderBy(col("token_id").cast("int")))

# ── Metadata Table ─────────────────────────────
display(spark.table("nft_metadata_table")
    .filter(col("contract_address") == contract)
    .orderBy(col("token_id").cast("int")))

# ── Rarity Table ───────────────────────────────
collection_id = spark.table("nft_base_table") \
    .filter(col("contract_address") == contract) \
    .select("collection_id") \
    .first()["collection_id"]

display(spark.table("nft_rarity_table")
    .filter(col("collection_id") == collection_id)
    .orderBy("rank"))

In [0]:
# ── Check what's in base table ─────────────────


print("=== COLLECTION REFERENCE ===")
display(spark.table("collection_reference"))